# SOMAX: Data Setup

**Step 1 of 2 — Environment, dependencies, and WAXAL dataset download**

Run this notebook once (or whenever you need fresh data). It:
1. Installs dependencies
2. Clones the somax repo
3. Downloads the WAXAL dataset
4. Backs up data to Google Drive

After this notebook completes, run `train_eval.ipynb`.

## 1. Setup & Dependencies

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Skip torch and psutil (already in Colab)
# Focus on the research-specific extensions
!uv pip install -U transformers datasets accelerate peft bitsandbytes llama-cpp-python

In [ ]:
import os
from pathlib import Path

if not os.path.exists('somax'):
    !git clone 'https://github.com/attabeezy/somax.git'
    %cd somax
else:
    %cd somax

In [ ]:
import os

# ── Configuration ─────────────────────────────────────────
LANGUAGE    = "akan"                      # proof-of-concept language
TRAIN_GROUP = "D"                         # primary hypothesis: TTS → ASR → TTS
BASE_MODEL  = "meta-llama/Llama-3.2-1B"
DATA_DIR    = f"data/{LANGUAGE}"
CONFIG_FILE = "configs/variants.yaml"
# ──────────────────────────────────────────────────────────

HF_TOKEN = os.environ.get("HF_TOKEN", "SIKE TOKEN NOT SET")
if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set. Required for Llama models.")
    print("Set it with:  os.environ['HF_TOKEN'] = 'your_token'")

In [ ]:
import shutil

def setup_drive() -> str | None:
    """Mount Google Drive and return results base path."""
    try:
        from google.colab import drive
        drive.mount("/content/drive", timeout_ms=120000)
        base = "/content/drive/MyDrive/WAXAL-results"
        os.makedirs(base, exist_ok=True)
        print(f"Google Drive mounted: {base}")
        return base
    except Exception as e:
        print(f"Google Drive not available: {e}")
        return None

def save_to_drive(source_dir: str, drive_base: str | None, label: str) -> None:
    """Copy a local directory to Google Drive."""
    if not drive_base or not os.path.exists(source_dir):
        return
    dest = os.path.join(drive_base, label)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(source_dir, dest)
    print(f"Saved to Drive: {dest}")

DRIVE_BASE = setup_drive()

## 2. Download WAXAL Dataset

In [ ]:
!python scripts/download.py --lang {LANGUAGE} --output data/

In [ ]:
save_to_drive(DATA_DIR, DRIVE_BASE, "data")